In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import sklearn.linear_model as lm
import sklearn.pipeline as pipe
import sklearn.preprocessing as pp
import sklearn.model_selection as ms
import sklearn.neighbors as n
import keras
import numpy as np

In [2]:
df = pd.read_csv("data/breast-cancer/data.csv")
df.describe()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst
count,5.690000e+02,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,...,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000
mean,3.037183e+07,0.372583,14.127292,19.289649,91.969033,654.889104,0.096360,0.104341,0.088799,0.048919,...,16.269190,25.677223,107.261213,880.583128,0.132369,0.254265,0.272188,0.114606,0.290076,0.083946
std,1.250206e+08,0.483918,3.524049,4.301036,24.298981,351.914129,0.014064,0.052813,0.079720,0.038803,...,4.833242,6.146258,33.602542,569.356993,0.022832,0.157336,0.208624,0.065732,0.061867,0.018061
min,8.670000e+03,0.000000,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,...,7.930000,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040
25%,8.692180e+05,0.000000,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.020310,...,13.010000,21.080000,84.110000,515.300000,0.116600,0.147200,0.114500,0.064930,0.250400,0.071460
50%,9.060240e+05,0.000000,13.370000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.033500,...,14.970000,25.410000,97.660000,686.500000,0.131300,0.211900,0.226700,0.099930,0.282200,0.080040
75%,8.813129e+06,1.000000,15.780000,21.800000,104.100000,782.700000,0.105300,0.130400,0.130700,0.074000,...,18.790000,29.720000,125.400000,1084.000000,0.146000,0.339100,0.382900,0.161400,0.317900,0.092080
max,9.113205e+08,1.000000,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,...,36.040000,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500


In [3]:
x = df.drop(["id","diagnosis"], axis=1)
y = df["diagnosis"]
x.shape, y.shape

((569, 30), (569,))

In [4]:
xtrain, xtest, ytrain, ytest = ms.train_test_split(x, y, train_size=0.8, test_size=0.2, random_state=42)
xtest.shape

(114, 30)

In [5]:
scaler = pp.RobustScaler()
scaler.fit(xtrain)
xtrain = scaler.transform(xtrain)
xtest = scaler.transform(xtest)
scaler.inverse_transform(xtest)

array([[1.247e+01, 1.860e+01, 8.109e+01, ..., 1.015e-01, 3.014e-01,
        8.750e-02],
       [1.894e+01, 2.131e+01, 1.236e+02, ..., 1.789e-01, 2.551e-01,
        6.589e-02],
       [1.546e+01, 1.948e+01, 1.017e+02, ..., 1.514e-01, 2.837e-01,
        8.019e-02],
       ...,
       [1.152e+01, 1.493e+01, 7.387e+01, ..., 9.608e-02, 2.664e-01,
        7.809e-02],
       [1.422e+01, 2.785e+01, 9.255e+01, ..., 8.219e-02, 1.890e-01,
        7.796e-02],
       [2.073e+01, 3.112e+01, 1.357e+02, ..., 1.659e-01, 2.868e-01,
        8.218e-02]], shape=(114, 30))

In [6]:
ytrain = keras.utils.to_categorical(ytrain)
ytest = keras.utils.to_categorical(ytest)
ytrain.shape

(455, 2)

In [7]:
model = keras.Sequential()
model.add(keras.layers.Input((xtrain.shape[1],)))
model.add(keras.layers.Dense(40, activation="relu"))
model.add(keras.layers.Dropout(0.2))
model.add(keras.layers.Dense(20, activation="relu"))
model.add(keras.layers.Dropout(0.1))
model.add(keras.layers.Dense(2, activation="softmax"))
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 40)                  │           1,240 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 40)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 20)                  │             820 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 20)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 2)                   │              42 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,102 (8.21 KB)

 Trainable params: 2,102 (8.21 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
model.compile(loss="categorical_crossentropy", metrics=["accuracy"])
model.fit(xtrain, ytrain, epochs=10, validation_split=0.2, batch_size=10)


Epoch 1/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6154 - loss: 0.5898 - val_accuracy: 0.9451 - val_loss: 0.3692
Epoch 2/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9148 - loss: 0.3087 - val_accuracy: 0.9780 - val_loss: 0.2086
Epoch 3/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9341 - loss: 0.2087 - val_accuracy: 0.9780 - val_loss: 0.1468
Epoch 4/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9451 - loss: 0.1542 - val_accuracy: 0.9780 - val_loss: 0.1248
Epoch 5/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9588 - loss: 0.1203 - val_accuracy: 0.9780 - val_loss: 0.1138
Epoch 6/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9533 - loss: 0.0968 - val_accuracy: 0.9780 - val_loss: 0.1054
Epoch 7/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9588 - loss: 0.1028 - val_accuracy: 0.9780 - val_loss: 0.1013
Epoch 8/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9753 - loss: 0.0895 - val_accuracy: 0.9780 - val_loss:

In [9]:
model.evaluate(xtest, ytest)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9737 - loss: 0.0653 


[0.06534375995397568, 0.9736841917037964]

In [10]:
model.save("data/breast-cancer/mlp.h5")

In [11]:
model.predict(xtest)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 


array([[7.46446609e-01, 2.53553361e-01],
       [6.87018546e-05, 9.99931335e-01],
       [2.61937710e-03, 9.97380555e-01],
       [9.93716538e-01, 6.28353935e-03],
       [9.99013424e-01, 9.86539642e-04],
       [2.48423365e-10, 1.00000000e+00],
       [3.05789598e-08, 1.00000000e+00],
       [4.08094041e-02, 9.59190607e-01],
       [4.32032436e-01, 5.67967534e-01],
       [9.99785364e-01, 2.14622967e-04],
       [9.79697406e-01, 2.03025471e-02],
       [8.06831662e-03, 9.91931736e-01],
       [9.98707891e-01, 1.29206362e-03],
       [1.03714123e-01, 8.96285951e-01],
       [9.99717891e-01, 2.82055553e-04],
       [1.77927432e-04, 9.99822080e-01],
       [9.99490976e-01, 5.08995668e-04],
       [9.99978662e-01, 2.13163712e-05],
       [9.99999523e-01, 4.32489429e-07],
       [4.79792607e-06, 9.99995232e-01],
       [8.97840559e-01, 1.02159426e-01],
       [9.79939878e-01, 2.00600605e-02],
       [2.07363904e-08, 1.00000000e+00],
       [9.99982834e-01, 1.72195469e-05],
       [9.993039

In [12]:
ypred > 0.5

NameError: name 'ypred' is not defined